# HoloQPI — model testing and evaluation

Runs the **actual trained checkpoints** of this project on real holograms and reports
everything the models produce, with the project's own preprocessing, measurement chain
and metric code imported directly from `holoqpi/` — nothing is reimplemented here.

**The pipeline under test**

    raw hologram (1024 px, 1 channel)
        -> centre crop to 900 px          data.align = center_crop
        -> z-score normalisation          data.hologram_normalisation = zscore
        -> MobileNetV2 encoder
            -> phase head          quantitative phase, radians
            -> amplitude head      transmitted amplitude   (only when the arm enables it)
            -> segmentation head   cell / background
        -> watershed instance labelling
        -> per cell: projected area, circularity, integrated phase,
                     optical volume, dry mass

**How to use it**

1. Edit **section 1** only — project root, which checkpoints to load, which samples to run.
2. Run all cells.
3. To test your own hologram, jump to **section 16**; a single `.tif` is enough.

Every step prints what it is doing. Nothing is cached between runs, so any cell can be
re-run on its own after section 3 has built the models.

---
## 1. Configuration — the only cell you normally edit

In [ ]:
# ============================================================================
#  CONFIGURATION
# ============================================================================

# Where the project root is, relative to this notebook (this file lives in test/).
PROJECT_ROOT = ".."

# Which trained arms to load. label -> config file.
# The checkpoint is found automatically at
# runs/<experiment_name>_<modality>/best_model.pt
MODELS = {
    "A_baseline":      "config/v2/a_baseline.yaml",           # the recommended configuration
    "KA_compact":      "config/v2/k_compact_a.yaml",          # 3.36 M params, the deployment model
    "D1_amplitude_fwd": "config/v2/d_forward_amplitude.yaml",  # has the AMPLITUDE head
}

# The arm used for the single-sample walkthrough in sections 5-10.
PRIMARY_MODEL = "A_baseline"

# Samples to evaluate.
#   None  -> discover every field present in data/phase
#   list  -> explicit stems, e.g. ["NCI_01", "T24_rotenone_500nM_50"]
SAMPLES = None

# Cap the batch section. None = no cap. Each forward pass is ~2 s on CPU, <0.1 s on GPU.
MAX_SAMPLES = None

# Restrict to one split from data/splits.json ("test" / "val" / "train"), or None for all.
# "test" is the honest default - it is the split the study reports. If the filter would
# leave no samples (a partial data copy, say) the notebook says so and falls back to
# everything present, rather than silently evaluating nothing.
SPLIT = "test"

# True re-enables holoqpi's own INFO logging. Off by default: the library logs to stderr
# and Jupyter interleaves that with stdout unpredictably, which scrambles the order of
# the progress messages. The facts worth seeing are printed explicitly either way.
VERBOSE_LOGGING = False

# "auto" | "cpu" | "cuda" | "cuda:2"
DEVICE = "auto"

# Where predictions, figures and tables are written.
OUTPUT_DIR = "test_outputs"

# Acquisition geometry: "off_axis" or "gabor".
MODALITY = "off_axis"

print("Configuration set.")
print(f"  project root    {PROJECT_ROOT}")
print(f"  models to load  {list(MODELS)}")
print(f"  primary model   {PRIMARY_MODEL}")
print(f"  modality        {MODALITY}")
print(f"  split filter    {SPLIT}")
print(f"  verbose logging {VERBOSE_LOGGING}")
print(f"  output dir      {OUTPUT_DIR}")

---
## 2. Environment and imports

Imports the project's own modules. If this cell fails, `PROJECT_ROOT` is wrong or the
`holoqpi` package is incomplete — it should contain 39 `.py` files.

In [ ]:
print("Setting up environment...")

import os, sys, json, time, warnings
from pathlib import Path

# Captured BEFORE the chdir below, so OUTPUT_DIR lands beside this notebook rather than
# in the project root - which is where a relative path would otherwise resolve.
NOTEBOOK_DIR = Path.cwd().resolve()

ROOT = Path(PROJECT_ROOT).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)                      # the configs use paths relative to the project root
print(f"  notebook directory {NOTEBOOK_DIR}")
print(f"  working directory  {Path.cwd()}")

import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "figure.facecolor": "white"})

import logging

def set_holoqpi_logging(verbose=False):
    """Route the library's loggers to stdout so their order matches the prints.

    holoqpi.utils.get_logger attaches a StreamHandler to stderr. Jupyter buffers stdout
    and stderr separately, so INFO lines land in the wrong place relative to print() --
    which is why model-loading messages appeared under the wrong model heading.
    """
    level = logging.INFO if verbose else logging.WARNING
    for name in ["holoqpi"] + [n for n in logging.root.manager.loggerDict
                               if n.startswith("holoqpi")]:
        lg = logging.getLogger(name)
        for h in list(lg.handlers):
            lg.removeHandler(h)
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("%(levelname)-7s | %(name)s | %(message)s"))
        lg.addHandler(h)
        lg.setLevel(level)
        lg.propagate = False


class LogCapture(logging.Handler):
    """Collect a logger's messages so they can be printed in the right order."""
    def __init__(self, logger_name):
        super().__init__()
        self.lines = []
        self.logger = logging.getLogger(logger_name)
    def emit(self, record):
        self.lines.append(record.getMessage())
    def __enter__(self):
        # Mute the logger's own handlers for the duration, or every captured line is
        # ALSO emitted normally and appears twice in the cell output.
        self._prev = self.logger.level
        self._muted = [(h, h.level) for h in self.logger.handlers]
        for handler, _ in self._muted:
            handler.setLevel(logging.CRITICAL)
        self.logger.setLevel(logging.INFO)
        self.logger.addHandler(self)
        return self
    def __exit__(self, *exc):
        self.logger.removeHandler(self)
        for handler, level in self._muted:
            handler.setLevel(level)
        self.logger.setLevel(self._prev)

print("Importing holoqpi...")
from holoqpi.config import load_config
from holoqpi.data import io as data_io
from holoqpi.data.masks import split_instances
from holoqpi.models import build_model
from holoqpi.engine import Evaluator, load_checkpoint, apply_learned_physics
from holoqpi.analysis.cells import calibration_from_config, match_cells, measure_cells
from holoqpi.metrics import (
    AmplitudeMetrics, MeasurementMetrics, PhaseMetrics, SegmentationMetrics,
)

set_holoqpi_logging(VERBOSE_LOGGING)

n_modules = len(list((ROOT / "holoqpi").rglob("*.py")))
print(f"  holoqpi modules found  {n_modules}  (expect 39)")

if DEVICE == "auto":
    DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    DEV = torch.device(DEVICE)

print()
print("Environment:")
print(f"  python       {sys.version.split()[0]}")
print(f"  torch        {torch.__version__}")
print(f"  numpy        {np.__version__}")
print(f"  matplotlib   {matplotlib.__version__}")
print(f"  CUDA available  {torch.cuda.is_available()}")
print(f"  device       {DEV}")
if DEV.type == "cuda":
    print(f"  GPU          {torch.cuda.get_device_name(DEV)}")

OUT = (NOTEBOOK_DIR / OUTPUT_DIR).resolve()
for folder in ("predictions", "figures", "tables"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)
print(f"  outputs      {OUT}")

# WHAT THIS RUN WROTE, recorded rather than inferred from a directory listing.
# A listing also returns figures left over from an earlier run with different settings,
# and a stale PNG is indistinguishable from a current one at a glance.
FIGURES_WRITTEN = {}
PRE_EXISTING = {p.name for p in (OUT / "figures").glob("*.png")}
if PRE_EXISTING:
    print(f"  NOTE: {len(PRE_EXISTING)} figure(s) already in {OUT/'figures'} from an "
          f"earlier run; they will be listed separately at the end.")

def save_fig(fig, name, dpi=140):
    path = OUT / "figures" / name
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    FIGURES_WRITTEN[name] = path
    print(f"  saved  figures/{name}")
    return path

---
## 3. Load the trained models

For each arm this builds the network from its config and restores `best_model.pt`.

Two details that matter and are easy to get wrong:

- **`apply_learned_physics` is called before `build_model`.** An arm trained with a free
  propagation distance stores the learned `z` in the checkpoint rather than the config, and
  the forward-model metric has to be built at the value that arm actually trained to.
- **`load_checkpoint` refuses a partial load.** If under 90 % of tensors match it raises,
  rather than leaving a partly random network that still emits plausible-looking masks.
  A healthy load prints `440/440 tensors matched`.

In [ ]:
print("Loading models...")
print()

REGISTRY = {}

for label, config_path in MODELS.items():
    print(f"Loading {label}  ({config_path})")
    cfg = load_config(config_path)

    # Honour the notebook's modality choice rather than the arm's default.
    cfg = cfg.merged({"data": {"modality": MODALITY}})

    run_dir = Path(cfg.paths.output_root) / f"{cfg.experiment_name}_{cfg.data.modality}"
    ckpt = run_dir / "best_model.pt"
    if not ckpt.is_file():
        print(f"  SKIPPED: no checkpoint at {ckpt}")
        print()
        continue

    # Must precede build_model - see the note above.
    cfg = apply_learned_physics(cfg, ckpt)

    with LogCapture("holoqpi.models.holonet") as built:
        model = build_model(cfg)
    with LogCapture("holoqpi.engine.trainer") as loaded:
        load_checkpoint(model, ckpt, DEV)
    model.to(DEV).eval()

    n_par = sum(p.numel() for p in model.parameters())
    has_amp = model.amplitude_head is not None

    REGISTRY[label] = {
        "label": label,
        "config_path": config_path,
        "cfg": cfg,
        "model": model,
        "run_dir": run_dir,
        "checkpoint": ckpt,
        "params": n_par,
        "has_amplitude": has_amp,
        "calibration": calibration_from_config(cfg),
    }
    print(f"  experiment       {cfg.experiment_name}")
    for line in built.lines + loaded.lines:
        print(f"  {line}")
    print(f"  parameters       {n_par/1e6:.2f} M")
    print(f"  amplitude head   {'ON' if has_amp else 'off'}")
    print(f"  front end        {cfg.model.frontend.kind}")
    print(f"  checkpoint       {ckpt}  ({ckpt.stat().st_size/1e6:.0f} MB)")
    print()

if not REGISTRY:
    raise SystemExit("No models loaded. Check MODELS and that runs/<exp>_<modality>/best_model.pt exists.")

if PRIMARY_MODEL not in REGISTRY:
    PRIMARY_MODEL = next(iter(REGISTRY))
    print(f"PRIMARY_MODEL was unavailable; using {PRIMARY_MODEL} instead.")

print(f"{len(REGISTRY)} model(s) ready: {list(REGISTRY)}")

# One config is used for data conventions (they are identical across arms).
BASE = REGISTRY[PRIMARY_MODEL]["cfg"]
CALIB = REGISTRY[PRIMARY_MODEL]["calibration"]

print()
print("Calibration in force (from config/base.yaml):")
print(f"  wavelength            {BASE.optics.wavelength_um} um")
print(f"  refraction increment  {BASE.optics.refraction_increment_ml_per_g} mL/g")
print(f"  pixel pitch           {BASE.optics.pixel_pitch_x_um} x {BASE.optics.pixel_pitch_y_um} um")
print(f"  pixel area            {CALIB.pixel_area_um2:.7f} um^2")
print(f"  dry mass prefactor    {CALIB.picogram_per_radian_um2:.6f} pg/(rad*um^2)")

---
## 4. Data loading

`load_sample()` reproduces the project's evaluation-time preprocessing exactly, by calling
the same `holoqpi.data.io` functions the training dataset uses:

| step | function | note |
|---|---|---|
| read phase reference | `read_phase_bin` | 900×900 float32 behind a 23-byte header |
| read hologram | `read_hologram` | 1024×1024 |
| align to the phase grid | `align_to_phase_grid` | **centre crop**, never resampling — interpolated fringes no longer encode optical path length |
| normalise for the network | `normalise_hologram` | z-score |

The raw hologram is kept **separately** from the normalised one: the network needs the
normalised input, while the physics forward model needs the intensity the sensor actually
recorded.

Every reference (phase, mask, amplitude) is **optional**. Pass a bare hologram and the
notebook runs inference and cell measurements, and simply skips the comparisons that need
ground truth.

This cell also defines the plotting helpers (`show`, `boundary_overlay`, `phase_scale`,
`symmetric_scale`), so every section below it can be re-run on its own.

In [ ]:
print("Discovering samples...")

DATA_ROOT = Path(BASE.paths.data_root)
HOLO_SUFFIX = BASE.formats.hologram.suffixes.get(MODALITY)
PHASE_SUFFIX = BASE.formats.phase_binary.suffix
MASK_SUFFIX = BASE.formats.mask.suffix
AMP_SUFFIX = BASE.formats.amplitude.suffix

def discover_stems():
    """Stems that have a hologram of the configured modality."""
    holo_dir = DATA_ROOT / (
        "off_axis_hologram" if MODALITY == "off_axis" else "in_line_gabor_hologram"
    )
    stems = sorted(p.name[: -len(HOLO_SUFFIX)] for p in holo_dir.glob(f"*{HOLO_SUFFIX}"))
    return stems

def split_stems(split):
    """Stems belonging to one split, or None when no filtering is possible.

    data/splits.json nests the lists under a top-level "splits" key alongside the seed
    and the stratification fields. Reading the split straight off the document returns
    an empty list, which would silently filter every sample away.
    """
    path = DATA_ROOT / BASE.paths.splits_file
    if split is None or not path.is_file():
        return None
    doc = json.loads(path.read_text())
    splits = doc.get("splits", doc)
    if split not in splits:
        print(f"  splits.json has no '{split}' key (found: {sorted(splits)})")
        return None
    return set(splits[split])

ALL_STEMS = discover_stems()
print(f"  holograms present  {len(ALL_STEMS)}")

keep = split_stems(SPLIT)
if keep is not None:
    filtered = [s for s in ALL_STEMS if s in keep]
    print(f"  '{SPLIT}' split declares {len(keep)} field(s); "
          f"{len(filtered)} of them are present here")
    if filtered:
        ALL_STEMS = filtered
    else:
        print(f"  WARNING: no field of the '{SPLIT}' split is present in {DATA_ROOT}.")
        print(f"           Falling back to all {len(ALL_STEMS)} hologram(s) found.")
        print( "           Numbers from fields the model TRAINED on are optimistic and")
        print( "           must not be quoted as test performance.")
elif SPLIT is not None:
    print(f"  '{SPLIT}' filter not applied (no usable splits file)")

STEMS = list(SAMPLES) if SAMPLES else ALL_STEMS
if MAX_SAMPLES:
    STEMS = STEMS[:MAX_SAMPLES]

print(f"  samples selected   {len(STEMS)}")
for s in STEMS[:8]:
    print(f"    {s}")
if len(STEMS) > 8:
    print(f"    ... and {len(STEMS)-8} more")


def load_sample(stem=None, hologram_path=None, phase_path=None,
                mask_path=None, amplitude_path=None, modality=None, quiet=False):
    """Load one field. Only a hologram is required; every reference is optional."""
    modality = modality or MODALITY
    holo_suffix = BASE.formats.hologram.suffixes.get(modality)

    if hologram_path is None:
        if stem is None:
            raise ValueError("give either stem= or hologram_path=")
        hologram_path = data_io.hologram_path(DATA_ROOT, BASE, stem, modality)
    hologram_path = Path(hologram_path)
    if stem is None:
        stem = hologram_path.name
        for suf in (holo_suffix, ".tif", ".tiff", ".png"):
            if suf and stem.endswith(suf):
                stem = stem[: -len(suf)]
                break
    if not quiet:
        print(f"  reading hologram   {hologram_path}")

    raw = data_io.read_hologram(hologram_path)
    aligned = data_io.align_to_phase_grid(raw, BASE.data.phase_size, BASE.data.align,
                                          data_io.hologram_crop_offset(BASE.data))
    normalised = data_io.normalise_hologram(aligned, BASE.data.hologram_normalisation)

    # --- optional phase reference -------------------------------------------
    phase_ref = None
    if phase_path is None:
        candidate = data_io.phase_path(DATA_ROOT, BASE, stem)
        phase_path = candidate if candidate.is_file() else None
    if phase_path is not None and Path(phase_path).is_file():
        phase_ref = data_io.read_phase_bin(Path(phase_path), BASE.formats.phase_binary).phase
        if not quiet:
            print(f"  reading phase ref  {phase_path}")

    # --- optional mask reference -------------------------------------------
    mask_ref = None
    if mask_path is None:
        candidate = data_io.mask_path(DATA_ROOT, BASE, stem)
        mask_path = candidate if candidate.is_file() else None
    if mask_path is not None and Path(mask_path).is_file():
        mask_ref = data_io.read_mask(Path(mask_path))
        if not quiet:
            print(f"  reading mask ref   {mask_path}")

    # --- optional amplitude reference --------------------------------------
    amp_ref = None
    if amplitude_path is None:
        candidate = DATA_ROOT / BASE.paths.amplitude_dir / f"{stem}{AMP_SUFFIX}"
        amplitude_path = candidate if candidate.is_file() else None
    if amplitude_path is not None and Path(amplitude_path).is_file():
        amp_ref = np.load(Path(amplitude_path)).astype(np.float32)
        if not quiet:
            print(f"  reading amp ref    {amplitude_path}")

    return {
        "stem": stem,
        "modality": modality,
        "hologram_raw_full": raw,
        "hologram_raw": aligned,
        "hologram": normalised.astype(np.float32),
        "phase_ref": None if phase_ref is None else phase_ref.astype(np.float32),
        "mask_ref": None if mask_ref is None else mask_ref,
        "amplitude_ref": amp_ref,
    }

print()
print("load_sample() ready.")

def show(ax, data, title, cmap="viridis", vmin=None, vmax=None, cbar=True):
    """Imshow one array, or a placeholder when it is absent."""
    if data is None:
        ax.text(0.5, 0.5, "not available", ha="center", va="center",
                transform=ax.transAxes, color="#888")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
        return None
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    if cbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    return im


def boundary_overlay(ax, mask_pred, mask_ref, background, title):
    """Predicted and reference cell outlines over the hologram."""
    from scipy import ndimage
    ax.imshow(background, cmap="gray")
    for m, colour in ((mask_ref, "#2ca02c"), (mask_pred, "#d62728")):
        if m is None:
            continue
        b = (m > 0).astype(np.uint8)
        edge = b - ndimage.binary_erosion(b, iterations=1).astype(np.uint8)
        rgba = np.zeros(b.shape + (4,))
        rgba[..., :3] = matplotlib.colors.to_rgb(colour)
        rgba[..., 3] = edge * 0.95
        ax.imshow(rgba)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def phase_scale(*arrays):
    """A shared 1-99.5 percentile colour range, so panels are comparable."""
    vals = np.concatenate([a.ravel() for a in arrays if a is not None])
    lo, hi = np.percentile(vals, [1, 99.5])
    return dict(vmin=float(lo), vmax=float(hi))


def symmetric_scale(data, pct=99):
    lim = float(np.percentile(np.abs(data), pct))
    return dict(cmap="RdBu_r", vmin=-lim, vmax=lim)

print("Plot helpers ready: show(), boundary_overlay(), phase_scale(), symmetric_scale().")

---
## 5. Single-sample inference

`run_model()` is the whole forward path: the normalised hologram becomes a
`(1, 1, 900, 900)` float tensor, and the network returns a dict. `phase` and
`segmentation` are always present; `amplitude` only for an arm whose amplitude head is on.

The segmentation head emits 2-class logits, so the mask is `argmax` over the channel axis —
the same reduction the project's evaluator uses.

In [ ]:
def run_model(entry, sample, quiet=False):
    """Run one arm on one sample. Returns predictions as numpy arrays."""
    model = entry["model"]
    x = torch.from_numpy(sample["hologram"])[None, None].float().to(DEV)

    t0 = time.time()
    with torch.no_grad():
        out = model(x)
    if DEV.type == "cuda":
        torch.cuda.synchronize(DEV)
    elapsed = time.time() - t0

    phase = out["phase"][0, 0].float().cpu().numpy()
    seg_logits = out["segmentation"][0].float().cpu().numpy()
    mask = out["segmentation"].argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
    amplitude = (out["amplitude"][0, 0].float().cpu().numpy()
                 if "amplitude" in out else None)

    if not quiet:
        print(f"  {entry['label']:<18} {elapsed*1000:7.0f} ms   "
              f"outputs: {sorted(out)}")

    return {
        "model": entry["label"],
        "stem": sample["stem"],
        "phase": phase,
        "mask": mask,
        "seg_logits": seg_logits,
        "amplitude": amplitude,
        "latency_ms": elapsed * 1000.0,
    }


print("Loading data...")
SAMPLE = load_sample(stem=STEMS[0])
print()
print(f"Sample: {SAMPLE['stem']}")
print(f"  hologram, as recorded   {SAMPLE['hologram_raw_full'].shape}  "
      f"range [{SAMPLE['hologram_raw_full'].min():.0f}, {SAMPLE['hologram_raw_full'].max():.0f}]")
print(f"  hologram, centre crop   {SAMPLE['hologram_raw'].shape}")
print(f"  hologram, normalised    {SAMPLE['hologram'].shape}  "
      f"mean {SAMPLE['hologram'].mean():+.3f}  sd {SAMPLE['hologram'].std():.3f}")
print(f"  phase reference         "
      f"{'absent' if SAMPLE['phase_ref'] is None else str(SAMPLE['phase_ref'].shape) + ' rad, range [%.2f, %.2f]' % (SAMPLE['phase_ref'].min(), SAMPLE['phase_ref'].max())}")
print(f"  mask reference          "
      f"{'absent' if SAMPLE['mask_ref'] is None else str(SAMPLE['mask_ref'].shape) + ', foreground %.1f%%' % (100*(SAMPLE['mask_ref']>0).mean())}")
print(f"  amplitude reference     "
      f"{'absent' if SAMPLE['amplitude_ref'] is None else str(SAMPLE['amplitude_ref'].shape)}")

print()
print("Running inference...")
PRED = {label: run_model(entry, SAMPLE) for label, entry in REGISTRY.items()}
P = PRED[PRIMARY_MODEL]

print()
print(f"Primary prediction ({PRIMARY_MODEL}):")
print(f"  phase        range [{P['phase'].min():+.3f}, {P['phase'].max():+.3f}] rad, mean {P['phase'].mean():+.4f}")
print(f"  segmentation foreground {100*(P['mask']>0).mean():.1f}%")
print(f"  amplitude    {'not produced by this arm' if P['amplitude'] is None else 'range [%.3f, %.3f], mean %.4f' % (P['amplitude'].min(), P['amplitude'].max(), P['amplitude'].mean())}")

---
## 6. Single-sample visualisation

Input, predictions, references and error maps on one page. The phase error panel is
**signed** and on a symmetric colour scale, because dry mass is an integral: a slow offset
inside a cell matters far more than zero-mean high-frequency error, and an absolute-error
map hides the sign that distinguishes them.

In [ ]:
print("Generating figure: single-sample panel...")

ref_phase = SAMPLE["phase_ref"]
pv = phase_scale(ref_phase if ref_phase is not None else P["phase"])

fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
fig.suptitle(f"{SAMPLE['stem']}  —  {PRIMARY_MODEL}  ({MODALITY})", fontsize=11, y=0.99)

show(axes[0, 0], SAMPLE["hologram_raw"], "raw hologram (900 px crop)", cmap="gray")
show(axes[0, 1], P["phase"], "PREDICTED phase [rad]", **pv)
show(axes[0, 2], ref_phase, "REFERENCE phase [rad]", **pv)
if ref_phase is not None:
    err = P["phase"] - ref_phase
    show(axes[0, 3], err, f"phase error (pred - ref)\nMAE {np.abs(err).mean():.4f} rad",
         **symmetric_scale(err))
else:
    show(axes[0, 3], None, "phase error")

show(axes[1, 0], P["mask"], "PREDICTED segmentation", cmap="gray", cbar=False)
show(axes[1, 1], SAMPLE["mask_ref"], "REFERENCE segmentation", cmap="gray", cbar=False)
boundary_overlay(axes[1, 2], P["mask"], SAMPLE["mask_ref"], SAMPLE["hologram_raw"],
                 "boundaries: red = pred, green = ref")

amp_entry = next((p for p in PRED.values() if p["amplitude"] is not None), None)
if amp_entry is not None:
    show(axes[1, 3], amp_entry["amplitude"],
         f"PREDICTED amplitude\n({amp_entry['model']})", cmap="magma")
else:
    show(axes[1, 3], None, "predicted amplitude\n(no arm with the head loaded)")

plt.tight_layout()
save_fig(fig, f"single_{SAMPLE['stem']}_{PRIMARY_MODEL}.png")
plt.show()

---
## 7. Phase metrics

Uses the project's `PhaseMetrics`. Errors are in **radians**, not normalised image-quality
scores, because the downstream measurement inherits them in physical units — a phase bias of
δ radians integrated over a cell propagates straight into its dry mass.

The in-cell / background split is the important part. Roughly four fifths of every field is
background, which is easy to reconstruct, so a field-wide MAE is dominated by the part of
the image dry mass never integrates. **`phase_mae_rad_in_cell` is the number a
measurement-readiness claim rests on.**

In [ ]:
print("Calculating phase metrics...")
print()

def phase_metrics(pred_phase, ref_phase, mask_ref):
    if ref_phase is None:
        return {}
    m = PhaseMetrics(BASE.evaluation.phase.psnr_data_range)
    m.update(pred_phase[None], ref_phase[None],
             mask=None if mask_ref is None else mask_ref[None])
    return m.compute()

rows = []
for label, p in PRED.items():
    r = phase_metrics(p["phase"], SAMPLE["phase_ref"], SAMPLE["mask_ref"])
    if r:
        r = {"model": label, **r}
        rows.append(r)

if rows:
    ph = pd.DataFrame(rows).set_index("model")
    keep = ["phase_mae_rad", "phase_rmse_rad", "phase_bias_rad", "phase_psnr_db",
            "phase_ssim", "phase_pearson_r", "phase_mae_rad_in_cell",
            "phase_bias_rad_in_cell", "phase_mae_rad_background"]
    ph = ph[[c for c in keep if c in ph.columns]]
    print(f"Phase metrics for {SAMPLE['stem']}:")
    print()
    print(ph.T.to_string(float_format=lambda v: f"{v:9.4f}"))
    print()
    print("  MAE / RMSE / bias are in radians. PSNR uses data_range = "
          f"{BASE.evaluation.phase.psnr_data_range} rad.")
    print("  in_cell >> background is expected and is the measurement-relevant figure.")
else:
    print("No phase reference for this sample - phase metrics skipped.")
    ph = pd.DataFrame()

---
## 8. Segmentation metrics

Uses the project's `SegmentationMetrics`: Dice, IoU, **Aggregated Jaccard Index** and
boundary F1 at a 2 px tolerance.

AJI is not a secondary metric here. Every reported quantity is per cell, so a prediction
that covers the right pixels but merges two touching cells yields two wrong measurements
despite an excellent Dice. Boundary F1 is the strictest of the four and is where the
learned model separates most clearly from classical reconstruction.

> **These labels are silver standard.** Every reference mask is
> `Otsu(gaussian * phase_reference)`, so segmentation is scored against a threshold of its
> own input. Dice here is agreement with that procedure, not with an independent annotation.

In [ ]:
print("Calculating segmentation metrics...")
print()

def seg_metrics(pred_mask, ref_mask):
    if ref_mask is None:
        return {}
    m = SegmentationMetrics(
        num_classes=BASE.model.segmentation_classes,
        boundary_tolerance=BASE.evaluation.segmentation.boundary_tolerance_px,
        instance_method=BASE.evaluation.segmentation.instance_from,
        watershed_min_distance=BASE.mask_generation.watershed_min_distance_px,
    )
    m.update(pred_mask[None], ref_mask[None])
    return m.compute()

rows = []
for label, p in PRED.items():
    r = seg_metrics(p["mask"], SAMPLE["mask_ref"])
    if r:
        rows.append({"model": label, **r})

if rows:
    sg = pd.DataFrame(rows).set_index("model")
    keep = ["seg_dice", "seg_iou", "seg_aji", "seg_boundary_f1", "seg_instance_count_mape"]
    sg = sg[[c for c in keep if c in sg.columns]]
    print(f"Segmentation metrics for {SAMPLE['stem']}:")
    print()
    print(sg.T.to_string(float_format=lambda v: f"{v:9.4f}"))
    print()
    print(f"  instance labelling: {BASE.evaluation.segmentation.instance_from}, "
          f"min distance {BASE.mask_generation.watershed_min_distance_px} px")
    print(f"  boundary tolerance: {BASE.evaluation.segmentation.boundary_tolerance_px} px")
else:
    print("No mask reference for this sample - segmentation metrics skipped.")
    sg = pd.DataFrame()

---
## 9. Cell-level quantitative measurements

This is what the project exists to produce. `measure_cells()` is the project's own function,
so the numbers here are identical to those in `runs/*/per_cell_test.csv`.

| quantity | key | unit |
|---|---|---|
| projected area | `area_um2` | µm² |
| circularity | `circularity` | — (4πA/P², 1 = perfect disc) |
| integrated phase / optical volume | `optical_volume_rad_um2` | rad·µm² |
| dry mass | `dry_mass_pg` | pg |
| mean / max phase | `mean_phase_rad`, `max_phase_rad` | rad |

Dry mass is **m = λ/(2πα) · Σφ·dx·dy**, a fixed scalar multiple of the integrated phase, so a
relative error in the enclosed phase integral *is* the relative error in dry mass. That
identity is why α and the pixel pitches cancel from every relative metric.

Predicted and reference cells are paired by greedy best IoU above
`match_iou_threshold = 0.5`. Only matched pairs enter the per-cell error statistics, which is
why detection recall must always be quoted beside them.

In [ ]:
print("Measuring cells...")
print()

MEAS_CFG = BASE.evaluation.measurement
INSTANCE_METHOD = BASE.evaluation.segmentation.instance_from
WATERSHED_D = BASE.mask_generation.watershed_min_distance_px

def cells_for(phase, mask, calibration=None):
    """Instance-label a mask and measure every accepted cell."""
    calibration = calibration or CALIB
    labels = split_instances((mask > 0).astype(np.uint8), INSTANCE_METHOD, WATERSHED_D)
    records = measure_cells(phase, mask, calibration, MEAS_CFG,
                            INSTANCE_METHOD, WATERSHED_D, labels=labels)
    return records, labels

print(f"  area filter  {MEAS_CFG.min_cell_area_um2} - {MEAS_CFG.max_cell_area_um2} um^2")
print(f"  match IoU    {MEAS_CFG.match_iou_threshold}")
print()

pred_cells, pred_labels = cells_for(P["phase"], P["mask"])
print(f"PREDICTED: {len(pred_cells)} cells accepted")

ref_cells, ref_labels = (None, None)
if SAMPLE["mask_ref"] is not None and SAMPLE["phase_ref"] is not None:
    ref_cells, ref_labels = cells_for(SAMPLE["phase_ref"], SAMPLE["mask_ref"])
    print(f"REFERENCE: {len(ref_cells)} cells accepted")

COLS = ["label", "area_um2", "circularity", "optical_volume_rad_um2",
        "dry_mass_pg", "mean_phase_rad", "max_phase_rad"]

print()
print("Predicted cells (first 10):")
pc = pd.DataFrame(pred_cells)
if not pc.empty:
    print(pc[COLS].head(10).to_string(index=False, float_format=lambda v: f"{v:10.3f}"))

if ref_cells is not None:
    print()
    print("Matching predicted to reference cells...")
    pairs = match_cells(pred_cells, ref_cells, pred_labels, ref_labels,
                        MEAS_CFG.match_iou_threshold)
    print(f"  matched pairs      {len(pairs)}")
    print(f"  missed references  {len(ref_cells) - len(pairs)}")
    print(f"  false positives    {len(pred_cells) - len(pairs)}")
    print(f"  detection recall   {len(pairs)/max(len(ref_cells),1):.4f}")
    print(f"  detection precision{len(pairs)/max(len(pred_cells),1):>8.4f}")

    if pairs:
        rows = []
        for pr, rf in pairs:
            row = {"match_iou": pr["match_iou"]}
            for k in ("area_um2", "circularity", "optical_volume_rad_um2", "dry_mass_pg"):
                row[f"{k}_pred"] = pr[k]
                row[f"{k}_ref"] = rf[k]
                row[f"{k}_relerr"] = (pr[k] - rf[k]) / rf[k] if rf[k] else np.nan
            rows.append(row)
        PAIRS = pd.DataFrame(rows)
        print()
        print("Per-cell errors on this field:")
        print()
        summary = []
        for k, unit in (("area_um2", "um^2"), ("circularity", "-"),
                        ("optical_volume_rad_um2", "rad*um^2"), ("dry_mass_pg", "pg")):
            rel = PAIRS[f"{k}_relerr"].abs()
            summary.append({
                "quantity": k, "unit": unit,
                "MAPE": rel.mean(), "median APE": rel.median(),
                "bias": PAIRS[f"{k}_relerr"].mean(),
                "pred mean": PAIRS[f"{k}_pred"].mean(),
                "ref mean": PAIRS[f"{k}_ref"].mean(),
            })
        CELLSUM = pd.DataFrame(summary).set_index("quantity")
        print(CELLSUM.to_string(float_format=lambda v: f"{v:10.4f}"))
        print()
        print("  MAPE and bias are dimensionless; 'pred mean'/'ref mean' carry the unit column.")
        print("  optical volume and dry mass differ only by the constant lambda/(2*pi*alpha),")
        print(f"  so their MAPEs are identical by construction ({CALIB.picogram_per_radian_um2:.6f} pg per rad*um^2).")
    else:
        PAIRS, CELLSUM = pd.DataFrame(), pd.DataFrame()
else:
    print()
    print("No reference phase+mask: reporting predicted cells only, no error statistics.")
    PAIRS, CELLSUM = pd.DataFrame(), pd.DataFrame()

In [ ]:
print("Generating figure: cell measurement agreement...")

if not PAIRS.empty:
    quantities = [("area_um2", "projected area [um^2]"),
                  ("dry_mass_pg", "dry mass [pg]"),
                  ("circularity", "circularity [-]")]
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    for j, (key, name) in enumerate(quantities):
        x, y = PAIRS[f"{key}_ref"], PAIRS[f"{key}_pred"]
        ax = axes[0, j]
        lim = [min(x.min(), y.min()), max(x.max(), y.max())]
        ax.plot(lim, lim, "k--", lw=1, alpha=0.6, label="identity")
        ax.scatter(x, y, s=18, alpha=0.75, edgecolor="none")
        ax.set_xlabel(f"reference {name}"); ax.set_ylabel(f"predicted {name}")
        r = np.corrcoef(x, y)[0, 1] if len(x) > 1 else np.nan
        ax.set_title(f"{name}\nPearson r = {r:.4f}", fontsize=9)
        ax.legend(fontsize=7)

        # Bland-Altman on the relative difference, which is how the project reports it.
        ax = axes[1, j]
        rel = PAIRS[f"{key}_relerr"]
        mean_val = (x + y) / 2
        bias, sd = rel.mean(), rel.std()
        ax.axhline(0, color="k", lw=0.8, alpha=0.5)
        ax.axhline(bias, color="#d62728", lw=1.2, label=f"bias {bias:+.3f}")
        ax.axhline(bias + 1.96*sd, color="#d62728", ls=":", lw=1,
                   label=f"LoA {bias-1.96*sd:+.3f} / {bias+1.96*sd:+.3f}")
        ax.axhline(bias - 1.96*sd, color="#d62728", ls=":", lw=1)
        ax.scatter(mean_val, rel, s=18, alpha=0.75, edgecolor="none")
        ax.set_xlabel(f"mean of pred and ref {name}")
        ax.set_ylabel("relative difference")
        ax.set_title("Bland-Altman (relative)", fontsize=9)
        ax.legend(fontsize=7)
    fig.suptitle(f"Cell measurement agreement — {SAMPLE['stem']} — {PRIMARY_MODEL} "
                 f"({len(PAIRS)} matched cells)", fontsize=11)
    plt.tight_layout()
    save_fig(fig, f"cells_{SAMPLE['stem']}_{PRIMARY_MODEL}.png")
    plt.show()
else:
    print("  skipped: no matched cell pairs on this field.")

---
## 10. Amplitude output

Only arms with `model.amplitude.enabled` produce an amplitude. `AmplitudeMetrics` scores it
against two comparators, and the second is the one that matters:

- the **reconstruction reference** (`amplitude_mae`)
- the **thin-phase-object assumption A = 1** (`amplitude_unity_mae`), which is what the rest
  of the study runs on

`amplitude_mae_over_unity` is their ratio. **Below 1.0** the head is closer to the reference
than assuming unit transmittance; a head that has collapsed to a constant scores **exactly
1.000** with `amplitude_pred_sd = 0`, which is the failure this comparator exists to expose.

> The reference is the modulus of a classical reconstruction, **not a measurement**. It
> carries the sideband filter's lost high frequencies, residual twin-image structure and any
> illumination vignetting. Every number here is agreement with one algorithm's output.

In [ ]:
print("Calculating amplitude metrics...")
print()

AMP_MODELS = [l for l, e in REGISTRY.items() if e["has_amplitude"]]
print(f"  arms with an amplitude head: {AMP_MODELS if AMP_MODELS else 'none loaded'}")

amp_rows = []
if AMP_MODELS and SAMPLE["amplitude_ref"] is not None:
    for label in AMP_MODELS:
        pred_amp = PRED[label]["amplitude"]
        if pred_amp is None:
            continue
        m = AmplitudeMetrics()
        m.update(pred_amp[None], SAMPLE["amplitude_ref"][None],
                 mask=None if SAMPLE["mask_ref"] is None else SAMPLE["mask_ref"][None])
        amp_rows.append({"model": label, **m.compute()})

if amp_rows:
    am = pd.DataFrame(amp_rows).set_index("model")
    keep = ["amplitude_mae", "amplitude_unity_mae", "amplitude_mae_over_unity",
            "amplitude_mae_in_cell", "amplitude_bias", "amplitude_pearson_r",
            "amplitude_pred_mean", "amplitude_pred_sd", "amplitude_reference_mean"]
    am = am[[c for c in keep if c in am.columns]]
    print()
    print(f"Amplitude metrics for {SAMPLE['stem']}:")
    print()
    print(am.T.to_string(float_format=lambda v: f"{v:9.4f}"))
    print()
    for label, row in am.iterrows():
        ratio = row["amplitude_mae_over_unity"]
        verdict = ("BETTER than assuming A = 1" if ratio < 1
                   else "NOT better than assuming A = 1")
        print(f"  {label}: ratio {ratio:.4f}  ->  {verdict}")
        if row["amplitude_pred_sd"] < 1e-6:
            print("    WARNING: predicted spread is zero - the head has collapsed.")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    a = PRED[AMP_MODELS[0]]["amplitude"]
    r = SAMPLE["amplitude_ref"]
    lo, hi = np.percentile(np.concatenate([a.ravel(), r.ravel()]), [1, 99])
    show(axes[0], a, f"PREDICTED amplitude ({AMP_MODELS[0]})", cmap="magma", vmin=lo, vmax=hi)
    show(axes[1], r, "REFERENCE amplitude (reconstruction)", cmap="magma", vmin=lo, vmax=hi)
    d = a - r
    lim = float(np.percentile(np.abs(d), 99))
    show(axes[2], d, f"difference\nMAE {np.abs(d).mean():.4f}", cmap="RdBu_r", vmin=-lim, vmax=lim)
    plt.tight_layout()
    save_fig(fig, f"amplitude_{SAMPLE['stem']}.png")
    plt.show()
else:
    am = pd.DataFrame()
    if not AMP_MODELS:
        print("  No amplitude arm loaded. Add config/v2/d0_amplitude.yaml, "
              "d_forward_amplitude.yaml or d2_learned_z.yaml to MODELS.")
    else:
        print("  No amplitude reference for this sample. Generate it with:")
        print("    python scripts/prepare_amplitude.py --config config/base.yaml")

---
## 11. Batch evaluation — every sample, every model

One row per (sample, model). This is the cell to re-run after changing `SAMPLES`,
`MAX_SAMPLES` or `SPLIT`.

Cost is roughly 2 s per forward on CPU and under 0.1 s on a GPU, plus about a second per
field for watershed labelling and the measurement chain.

In [ ]:
print("Running batch evaluation...")
print(f"  {len(STEMS)} sample(s) x {len(REGISTRY)} model(s) = {len(STEMS)*len(REGISTRY)} inference passes")
print()

BATCH_ROWS = []
BATCH_CELLS = []
PRED_CACHE = {}

t_start = time.time()
for i, stem in enumerate(STEMS, 1):
    print(f"[{i}/{len(STEMS)}] {stem}")
    s = load_sample(stem=stem, quiet=True)
    has_ref = s["phase_ref"] is not None and s["mask_ref"] is not None

    ref_cells, ref_labels = (None, None)
    if has_ref:
        ref_cells, ref_labels = cells_for(s["phase_ref"], s["mask_ref"])

    for label, entry in REGISTRY.items():
        p = run_model(entry, s, quiet=True)
        PRED_CACHE[(stem, label)] = p

        row = {"stem": stem, "model": label, "latency_ms": p["latency_ms"],
               "params_M": entry["params"] / 1e6}
        row.update(phase_metrics(p["phase"], s["phase_ref"], s["mask_ref"]))
        row.update(seg_metrics(p["mask"], s["mask_ref"]))

        pcells, plabels = cells_for(p["phase"], p["mask"], entry["calibration"])
        row["cells_predicted"] = len(pcells)

        if entry["has_amplitude"] and p["amplitude"] is not None and s["amplitude_ref"] is not None:
            m = AmplitudeMetrics()
            m.update(p["amplitude"][None], s["amplitude_ref"][None],
                     mask=None if s["mask_ref"] is None else s["mask_ref"][None])
            row.update(m.compute())

        if has_ref:
            row["cells_reference"] = len(ref_cells)
            pairs = match_cells(pcells, list(ref_cells), plabels, ref_labels,
                                MEAS_CFG.match_iou_threshold)
            row["cells_matched"] = len(pairs)
            row["detection_recall"] = len(pairs) / max(len(ref_cells), 1)
            row["detection_precision"] = len(pairs) / max(len(pcells), 1)
            denom = row["detection_recall"] + row["detection_precision"]
            row["detection_f1"] = (2 * row["detection_recall"] * row["detection_precision"] / denom
                                   if denom else 0.0)
            for key in ("area_um2", "circularity", "optical_volume_rad_um2", "dry_mass_pg"):
                rel = [abs(pr[key] - rf[key]) / rf[key] for pr, rf in pairs if rf[key]]
                sgn = [(pr[key] - rf[key]) / rf[key] for pr, rf in pairs if rf[key]]
                row[f"{key}_mape"] = float(np.mean(rel)) if rel else np.nan
                row[f"{key}_bias"] = float(np.mean(sgn)) if sgn else np.nan
            for pr, rf in pairs:
                BATCH_CELLS.append({
                    "stem": stem, "model": label, "match_iou": pr["match_iou"],
                    **{f"{k}_pred": pr[k] for k in
                       ("area_um2", "circularity", "optical_volume_rad_um2", "dry_mass_pg")},
                    **{f"{k}_ref": rf[k] for k in
                       ("area_um2", "circularity", "optical_volume_rad_um2", "dry_mass_pg")},
                })
        BATCH_ROWS.append(row)

    done = ", ".join(f"{l}: {PRED_CACHE[(stem,l)]['latency_ms']:.0f} ms" for l in REGISTRY)
    print(f"      {done}")

RESULTS = pd.DataFrame(BATCH_ROWS)
CELLS_ALL = pd.DataFrame(BATCH_CELLS)

print()
print(f"Batch complete in {time.time()-t_start:.1f} s")
print(f"  result rows   {len(RESULTS)}")
print(f"  matched cells {len(CELLS_ALL)}")
print(f"  columns       {len(RESULTS.columns)}")

### Per-sample results

In [ ]:
print("Generating per-sample table...")
print()

VIEW = ["stem", "model", "phase_mae_rad", "phase_mae_rad_in_cell", "phase_pearson_r",
        "seg_dice", "seg_aji", "seg_boundary_f1", "detection_recall",
        "area_um2_mape", "dry_mass_pg_mape", "circularity_mape",
        "cells_matched", "cells_reference", "latency_ms"]
view = [c for c in VIEW if c in RESULTS.columns]

for label in RESULTS["model"].unique():
    sub = RESULTS[RESULTS["model"] == label][view].drop(columns=["model"])
    print(f"--- {label} " + "-" * max(0, 60 - len(label)))
    print(sub.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
    print()

---
## 12. Summary — mean, standard deviation, median

Read the matched-cell MAPE and the detection recall **together**. Per-cell errors are
computed over IoU-matched pairs only, so a detector that finds the easiest cells and
measures them perfectly scores well on the first while saying nothing about the cells it
never reported.

In [ ]:
print("Generating summary statistics...")
print()

NUMERIC = [c for c in RESULTS.columns
           if c not in ("stem", "model") and pd.api.types.is_numeric_dtype(RESULTS[c])]

SUMMARY = RESULTS.groupby("model")[NUMERIC].agg(["mean", "std", "median"])

HEADLINE = [c for c in ["phase_mae_rad", "phase_mae_rad_in_cell", "phase_rmse_rad",
                        "phase_ssim", "phase_pearson_r", "seg_dice", "seg_iou", "seg_aji",
                        "seg_boundary_f1", "detection_recall", "detection_precision",
                        "detection_f1", "area_um2_mape", "dry_mass_pg_mape",
                        "optical_volume_rad_um2_mape", "circularity_mape",
                        "amplitude_mae_over_unity", "latency_ms"]
            if c in NUMERIC]

for label in RESULTS["model"].unique():
    sub = RESULTS[RESULTS["model"] == label]
    print(f"=== {label}   (n = {len(sub)} fields) " + "=" * max(0, 40 - len(label)))
    tab = pd.DataFrame({
        "mean":   sub[HEADLINE].mean(),
        "std":    sub[HEADLINE].std(),
        "median": sub[HEADLINE].median(),
        "min":    sub[HEADLINE].min(),
        "max":    sub[HEADLINE].max(),
    })
    print(tab.to_string(float_format=lambda v: f"{v:10.4f}"))
    print()

print("Note: 'std' here is the spread ACROSS FIELDS for one model, not between training")
print("seeds. The study's resolution criterion (2 x pooled between-seed SD) needs several")
print("seeds per arm and is applied in scripts/aggregate_seeds.py, not here.")

---
## 13. Model-to-model comparison

Same fields, same preprocessing, same metric code — so a difference between rows is a
difference between the trained models and nothing else.

In [ ]:
print("Comparing models...")
print()

COMPARE = RESULTS.groupby("model")[HEADLINE].mean()
COMPARE.insert(0, "params_M", RESULTS.groupby("model")["params_M"].first())

print(f"Mean over {RESULTS['stem'].nunique()} field(s):")
print()
print(COMPARE.T.to_string(float_format=lambda v: f"{v:10.4f}"))

if len(COMPARE) > 1 and PRIMARY_MODEL in COMPARE.index:
    print()
    print(f"Signed difference against {PRIMARY_MODEL} (negative = lower value):")
    print()
    delta = COMPARE.drop(columns=["params_M"]).sub(
        COMPARE.drop(columns=["params_M"]).loc[PRIMARY_MODEL], axis=1)
    print(delta.drop(index=PRIMARY_MODEL).T.to_string(float_format=lambda v: f"{v:+10.4f}"))
    print()
    print("  Lower is better for: MAE, RMSE, every *_mape, latency.")
    print("  Higher is better for: SSIM, Pearson r, Dice, IoU, AJI, boundary F1, recall.")

In [ ]:
print("Generating figure: model comparison...")

PLOT_METRICS = [("phase_mae_rad", "phase MAE [rad]", "lower"),
                ("phase_mae_rad_in_cell", "in-cell phase MAE [rad]", "lower"),
                ("seg_dice", "Dice", "higher"),
                ("seg_aji", "AJI", "higher"),
                ("seg_boundary_f1", "boundary F1", "higher"),
                ("dry_mass_pg_mape", "dry mass MAPE", "lower"),
                ("area_um2_mape", "area MAPE", "lower"),
                ("detection_recall", "detection recall", "higher")]
PLOT_METRICS = [m for m in PLOT_METRICS if m[0] in RESULTS.columns]

if PLOT_METRICS:
    ncol = 4
    nrow = int(np.ceil(len(PLOT_METRICS) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.3*ncol, 3.0*nrow), squeeze=False)
    labels = list(COMPARE.index)
    colours = plt.cm.tab10(np.linspace(0, 1, 10))[:len(labels)]
    for k, (key, name, better) in enumerate(PLOT_METRICS):
        ax = axes[k // ncol][k % ncol]
        means = [RESULTS[RESULTS["model"] == l][key].mean() for l in labels]
        sds = [RESULTS[RESULTS["model"] == l][key].std() for l in labels]
        ax.bar(range(len(labels)), means, yerr=sds, capsize=3,
               color=colours, edgecolor="k", linewidth=0.5)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=7)
        ax.set_title(f"{name}\n({better} is better)", fontsize=8.5)
        ax.grid(axis="y", alpha=0.3)
    for k in range(len(PLOT_METRICS), nrow*ncol):
        axes[k // ncol][k % ncol].axis("off")
    fig.suptitle(f"Model comparison — mean over {RESULTS['stem'].nunique()} field(s), "
                 f"error bars = SD across fields", fontsize=11)
    plt.tight_layout()
    save_fig(fig, "model_comparison.png")
    plt.show()
else:
    print("  skipped: no reference data, so no comparable metrics.")

---
## 14. Grid visualisations

**Grid A** — one row per sample for the primary model: hologram, predicted phase, reference
phase, signed error, predicted and reference segmentation.

**Grid B** — one row per model on a single field, so architectures can be compared on
identical input.

In [ ]:
print("Generating figure: grid A (samples x outputs)...")

GRID_N = min(6, len(STEMS))
stems_grid = STEMS[:GRID_N]
cols = ["hologram", "pred phase", "ref phase", "phase error", "pred seg", "ref seg"]

fig, axes = plt.subplots(GRID_N, len(cols), figsize=(2.5*len(cols), 2.5*GRID_N), squeeze=False)
for i, stem in enumerate(stems_grid):
    s = load_sample(stem=stem, quiet=True)
    p = PRED_CACHE[(stem, PRIMARY_MODEL)]
    ref = s["phase_ref"]
    if ref is not None:
        lo, hi = np.percentile(ref, [1, 99.5])
    else:
        lo, hi = np.percentile(p["phase"], [1, 99.5])

    show(axes[i][0], s["hologram_raw"], "", cmap="gray", cbar=False)
    show(axes[i][1], p["phase"], "", vmin=lo, vmax=hi, cbar=False)
    show(axes[i][2], ref, "", vmin=lo, vmax=hi, cbar=False)
    if ref is not None:
        e = p["phase"] - ref
        lim = float(np.percentile(np.abs(e), 99))
        show(axes[i][3], e, "", cmap="RdBu_r", vmin=-lim, vmax=lim, cbar=False)
    else:
        show(axes[i][3], None, "")
    show(axes[i][4], p["mask"], "", cmap="gray", cbar=False)
    show(axes[i][5], s["mask_ref"], "", cmap="gray", cbar=False)

    axes[i][0].set_ylabel(stem, fontsize=7, rotation=0, ha="right", va="center")
    axes[i][0].axis("on")
    axes[i][0].set_xticks([]); axes[i][0].set_yticks([])

for j, name in enumerate(cols):
    axes[0][j].set_title(name, fontsize=9)

fig.suptitle(f"Grid A — {PRIMARY_MODEL}, {GRID_N} field(s), phase in radians", fontsize=11)
plt.tight_layout()
save_fig(fig, "grid_samples.png", dpi=130)
plt.show()

In [ ]:
print("Generating figure: grid B (models x outputs, one field)...")

stem = STEMS[0]
s = load_sample(stem=stem, quiet=True)
cols = ["pred phase", "phase error", "pred seg", "pred amplitude"]
labels = list(REGISTRY)

fig, axes = plt.subplots(len(labels), len(cols),
                         figsize=(3.0*len(cols), 2.9*len(labels)), squeeze=False)
ref = s["phase_ref"]
lo, hi = (np.percentile(ref, [1, 99.5]) if ref is not None
          else np.percentile(PRED_CACHE[(stem, labels[0])]["phase"], [1, 99.5]))

for i, label in enumerate(labels):
    p = PRED_CACHE[(stem, label)]
    show(axes[i][0], p["phase"], "", vmin=lo, vmax=hi, cbar=False)
    if ref is not None:
        e = p["phase"] - ref
        lim = float(np.percentile(np.abs(e), 99))
        show(axes[i][1], e, f"MAE {np.abs(e).mean():.4f}", cmap="RdBu_r",
             vmin=-lim, vmax=lim, cbar=False)
    else:
        show(axes[i][1], None, "")
    show(axes[i][2], p["mask"], "", cmap="gray", cbar=False)
    show(axes[i][3], p["amplitude"], "", cmap="magma", cbar=False)

    axes[i][0].set_ylabel(f"{label}\n{REGISTRY[label]['params']/1e6:.2f} M",
                          fontsize=7.5, rotation=0, ha="right", va="center")
    axes[i][0].axis("on"); axes[i][0].set_xticks([]); axes[i][0].set_yticks([])

for j, name in enumerate(cols):
    axes[0][j].set_title(name, fontsize=9)

fig.suptitle(f"Grid B — every model on {stem}", fontsize=11)
plt.tight_layout()
save_fig(fig, "grid_models.png", dpi=130)
plt.show()

---
## 15. Save predictions, figures and results

- `predictions/<stem>_<model>.npz` — phase, mask, amplitude, and the references present
- `tables/per_sample_results.csv` — one row per (sample, model), every metric
- `tables/per_cell_measurements.csv` — one row per matched cell
- `tables/summary.csv`, `tables/model_comparison.csv`
- `tables/run_metadata.json` — device, checkpoints, calibration, config paths
- `figures/*.png`

`.npz` keeps full float precision. Load one with
`np.load("predictions/NCI_01_A_baseline.npz")`.

In [ ]:
print("Saving results...")
print()

print("Saving predictions...")
n = 0
for (stem, label), p in PRED_CACHE.items():
    s = load_sample(stem=stem, quiet=True)
    payload = {"phase_pred": p["phase"], "mask_pred": p["mask"],
               "hologram_raw": s["hologram_raw"], "hologram_norm": s["hologram"]}
    if p["amplitude"] is not None:
        payload["amplitude_pred"] = p["amplitude"]
    if s["phase_ref"] is not None:
        payload["phase_ref"] = s["phase_ref"]
    if s["mask_ref"] is not None:
        payload["mask_ref"] = s["mask_ref"]
    if s["amplitude_ref"] is not None:
        payload["amplitude_ref"] = s["amplitude_ref"]
    np.savez_compressed(OUT / "predictions" / f"{stem}_{label}.npz", **payload)
    n += 1
print(f"  {n} .npz file(s) -> {OUT/'predictions'}")

print("Saving tables...")
RESULTS.to_csv(OUT / "tables" / "per_sample_results.csv", index=False)
print(f"  per_sample_results.csv      {RESULTS.shape[0]} rows x {RESULTS.shape[1]} cols")
if not CELLS_ALL.empty:
    CELLS_ALL.to_csv(OUT / "tables" / "per_cell_measurements.csv", index=False)
    print(f"  per_cell_measurements.csv   {CELLS_ALL.shape[0]} rows")
SUMMARY.to_csv(OUT / "tables" / "summary.csv")
print("  summary.csv")
COMPARE.to_csv(OUT / "tables" / "model_comparison.csv")
print("  model_comparison.csv")

META = {
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": str(DEV),
    "cuda_available": torch.cuda.is_available(),
    "torch": torch.__version__,
    "modality": MODALITY,
    "split_filter": SPLIT,
    "n_samples": len(STEMS),
    "samples": STEMS,
    "models": {
        label: {
            "config": e["config_path"],
            "experiment_name": e["cfg"].experiment_name,
            "checkpoint": str(e["checkpoint"]),
            "parameters": int(e["params"]),
            "amplitude_head": bool(e["has_amplitude"]),
            "frontend": e["cfg"].model.frontend.kind,
        } for label, e in REGISTRY.items()
    },
    "calibration": {
        "wavelength_um": BASE.optics.wavelength_um,
        "refraction_increment_ml_per_g": BASE.optics.refraction_increment_ml_per_g,
        "pixel_pitch_x_um": BASE.optics.pixel_pitch_x_um,
        "pixel_pitch_y_um": BASE.optics.pixel_pitch_y_um,
        "pixel_area_um2": CALIB.pixel_area_um2,
        "pg_per_radian_um2": CALIB.picogram_per_radian_um2,
    },
    "preprocessing": {
        "phase_size": BASE.data.phase_size,
        "align": BASE.data.align,
        "hologram_normalisation": BASE.data.hologram_normalisation,
    },
    "evaluation": {
        "match_iou_threshold": MEAS_CFG.match_iou_threshold,
        "min_cell_area_um2": MEAS_CFG.min_cell_area_um2,
        "max_cell_area_um2": MEAS_CFG.max_cell_area_um2,
        "instance_from": INSTANCE_METHOD,
        "watershed_min_distance_px": WATERSHED_D,
        "boundary_tolerance_px": BASE.evaluation.segmentation.boundary_tolerance_px,
        "psnr_data_range": BASE.evaluation.phase.psnr_data_range,
    },
}
(OUT / "tables" / "run_metadata.json").write_text(json.dumps(META, indent=2))
print("  run_metadata.json")

print(f"Figures written by THIS run: {len(FIGURES_WRITTEN)}")
for name, path in sorted(FIGURES_WRITTEN.items()):
    print(f"  {name}  ({path.stat().st_size/1024:.0f} KB)")

stale = PRE_EXISTING - set(FIGURES_WRITTEN)
if stale:
    print()
    print(f"{len(stale)} figure(s) in {OUT/'figures'} are from an EARLIER run and were not")
    print("regenerated now. They describe different settings and are not results of this run:")
    for name in sorted(stale):
        print(f"  {name}")
    print("Delete the output folder before a clean run if that is confusing.")

META["figures_written"] = sorted(FIGURES_WRITTEN)
META["figures_stale"] = sorted(stale)
(OUT / "tables" / "run_metadata.json").write_text(json.dumps(META, indent=2))

print()
print(f"Done. Everything is under {OUT}")

---
## 16. Using your own data

### A single file

Only a hologram is needed. Every reference is optional, and whatever is missing is simply
skipped.

```python
s = load_sample(hologram_path="/path/to/my_hologram.tif")
p = run_model(REGISTRY[PRIMARY_MODEL], s)
cells, labels = cells_for(p["phase"], p["mask"])
print(pd.DataFrame(cells)[["area_um2", "circularity", "dry_mass_pg"]].describe())
```

With references, when their filenames do not follow the project's conventions:

```python
s = load_sample(
    hologram_path="my_holo.tif",
    phase_path="my_phase.bin",      # 900x900 float32 behind a 23-byte header
    mask_path="my_mask.png",        # 0 = background, non-zero = cell
    amplitude_path="my_amp.npy",
)
```

### Several files

```python
MY_FILES = ["a.tif", "b.tif", "c.tif"]
rows = []
for f in MY_FILES:
    s = load_sample(hologram_path=f, quiet=True)
    p = run_model(REGISTRY[PRIMARY_MODEL], s, quiet=True)
    cells, _ = cells_for(p["phase"], p["mask"])
    rows.append({"file": f, "cells": len(cells),
                 "total_dry_mass_pg": sum(c["dry_mass_pg"] for c in cells)})
print(pd.DataFrame(rows))
```

### Files that follow the project layout

Drop them into `data/off_axis_hologram/` as `<stem>_holo.tif`, then set
`SAMPLES = ["<stem>", ...]` in section 1 and re-run.

### Three things that will bite you

1. **Input size.** The loader centre-crops to 900 px and never resamples — interpolated
   fringes no longer encode optical path length. A hologram **smaller** than 900 px will
   fail; pad it, or set `data.phase_size` to match your sensor and retrain.
2. **Modality.** An off-axis checkpoint on an in-line hologram produces confident nonsense.
   Set `MODALITY` to match the instrument, and load `config/v2/g_baseline_gabor.yaml` for
   in-line data.
3. **Absolute units.** λ, α and the pixel pitch come from `config/base.yaml` and are specific
   to this instrument. Relative metrics are invariant to them; **absolute picograms and µm²
   are not.** Change them there, never in code.